# TAD case studies and confounder checks

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig4_S19_S20_S22_S23_neurips2021_cite_bmmc_fc_interaction_gene_layer_tad_analysis")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from collections import defaultdict
import sklearn
from pysankey import sankey
from sklearn.preprocessing import normalize
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt
import os
import pyBigWig
from scipy.stats import spearmanr
import ast
import glob
from pathlib import Path
from pybiomart import Server
import math


In [ ]:
obodag = GODag(
    os.path.join(
        os.path.abspath("../.experiment_data/primary"), "preprocessing", "resources", "ebi_ensembl_go_mappings", "go-basic.obo"
    )
)


In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, False, True)


In [ ]:
phenotypes = [
    "HSC",
    "Lymph_prog",
    "G_M_prog",
    "MK_E_prog",
    "ILC",
    "ILC1",
    "NK",
    "NK_CD158e1+",
    "CD4+_T_naive",
    "CD4+_T_activated",
    "CD4+_T_activated_integrinB7+",
    "CD4+_T_CD314+_CD45RA+",
    "CD8+_T_naive",
    "CD8+_T_naive_CD127+_CD26-_CD101-",
    "T_prog_cycling",
    "CD8+_T_CD49f+",
    "CD8+_T_CD69+_CD45RA+",
    "CD8+_T_CD69+_CD45RO+",
    "CD8+_T_TIGIT+_CD45RA+",
    "CD8+_T_TIGIT+_CD45RO+",
    "CD8+_T_CD57+_CD45RA+",
    "CD8+_T_CD57+_CD45RO+",
    "gdT_CD158b+",
    "gdT_TCRVD2+",
    "dnT",
    "T_reg",
    "Naive_CD20+_B_IGKC+",
    "Naive_CD20+_B_IGKC-",
    "Transitional_B",
    "B1_B_IGKC+",
    "B1_B_IGKC-",
    "Plasma_cell_IGKC+",
    "Plasma_cell_IGKC-",
    "Plasmablast_IGKC+",
    "Plasmablast_IGKC-",
    "cDC1",
    "cDC2",
    "pDC",
    "CD14+_Mono",
    "CD16+_Mono",
    "Erythroblast",
    "Normoblast",
    "Proerythroblast",
    "Reticulocyte",
    "MAIT",
]
phenotypes_labels = [
    "HSC",
    "Lymph prog",
    "G/M prog",
    "MK/E prog",
    "ILC",
    "ILC1",
    "NK",
    "NK CD158e1+",
    "CD4+ T naive",
    "CD4+ T activated",
    "CD4+ T activated B7+",
    "CD4+ T CD314+ CD45RA+",
    "CD8+ T naive",
    "CD8+ T naive CD127+ CD26- CD101-",
    "T prog cycling",
    "CD8+ T CD49f+",
    "CD8+ T CD69+ CD45RA+",
    "CD8+ T CD69+ CD45RO+",
    "CD8+ T TIGIT+ CD45RA+",
    "CD8+ T TIGIT+ CD45RO+",
    "CD8+ T CD57+ CD45RA+",
    "CD8+ T CD57+ CD45RO+",
    "gdT CD158b+",
    "gdT TCRVD2+",
    "dnT",
    "T reg",
    "Naive CD20+ B IGKC+",
    "Naive CD20+ B IGKC-",
    "Transitional B",
    "B1 B IGKC+",
    "B1 B IGKC-",
    "Plasma cell IGKC+",
    "Plasma cell IGKC-",
    "Plasmablast IGKC+",
    "Plasmablast IGKC-",
    "cDC1",
    "cDC2",
    "pDC",
    "CD14+ Mono",
    "CD16+ Mono",
    "Erythroblast",
    "Normoblast",
    "Proerythroblast",
    "Reticulocyte",
    "MAIT",
]
phenotypes_dataset = phenotypes_labels


In [ ]:
import pandas as pd

_GENE_POSITION_MAPPING_PATH = os.path.abspath(
    os.path.join(
        "..", "preprocessing", "resources", "ensembl_gene_id_chromosome_position_mapping",
        "ensembl_gene_id_chromosome_position_mapping.csv",
    )
)

def _get_ensembl_gene_id_chromosome_name_start_position_end_position_mapping():
    return pd.read_csv(_GENE_POSITION_MAPPING_PATH)


In [ ]:
def aggregate_dataframes(graph_csv_list):
    num_rows = len(graph_csv_list[0])
    for df in graph_csv_list:
        if len(df) != num_rows:
            raise ValueError("Alle DataFrames müssen die gleiche Anzahl an Zeilen haben.")

    aggregated_roc = []
    aggregated_activations = []

    for row_idx in range(num_rows):

        roc_values = [df.iloc[row_idx]["predictors_label_valid_roc"] for df in graph_csv_list]

        mean_roc = np.mean(roc_values, axis=0)
        aggregated_roc.append(mean_roc)

        activations_arrays = [df.iloc[row_idx]["activations_valid"] for df in graph_csv_list]

        concatenated_activations = np.concatenate(activations_arrays, axis=1)

        mean_activations = np.mean(concatenated_activations, axis=1)
        aggregated_activations.append(mean_activations)

    aggregated_df = pd.DataFrame(
        index=graph_csv_list[0].index,
        data={
            "predictors_label_valid_roc": aggregated_roc,
            "activations_valid": aggregated_activations,
        },
    )

    return aggregated_df


## Gene Groups

### Loading

In [ ]:
seeds = ["0"]

graph_csv_list = []
graph_csv_index_list = []
gene_groups_importances = []
for modality in tqdm(["expression", "protein"]):
    gene_groups_importances_modality = []
    graph_csv_modality_list = []
    for phenotype in tqdm([x for _, x in sorted(zip(phenotypes_labels, phenotypes))]):
        graph_csv_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_gene_groups_ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_gene_groups_evaluated_graph_csv.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        graph_csv_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_list)
        graph_csv_modality_list.append(graph_csv)
        gene_groups_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_list.append(graph_csv_modality_list)
    graph_csv_index_list.append(np.array(graph_csv_modality_list[0].index))
    gene_groups_importances.append(gene_groups_importances_modality)


In [ ]:
seeds = ["0"]

graph_csv_baseline_list = []
graph_csv_index_baseline_list = []
gene_groups_baseline_importances = []
for modality in tqdm(["expression", "protein"]):
    gene_groups_importances_modality = []
    graph_csv_modality_list = []
    for phenotype in tqdm([x for _, x in sorted(zip(phenotypes_labels, phenotypes))]):
        graph_csv_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_gene_groups_ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_gene_groups_evaluated_graph_csv.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        graph_csv_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_list)
        graph_csv_modality_list.append(graph_csv)
        gene_groups_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_baseline_list.append(graph_csv_modality_list)
    graph_csv_index_baseline_list.append(np.array(graph_csv_modality_list[0].index))
    gene_groups_baseline_importances.append(gene_groups_importances_modality)


## GO

In [ ]:
phenotypes = [
    "HSC",
    "Lymph_prog",
    "G_M_prog",
    "MK_E_prog",
    "ILC",
    "ILC1",
    "NK",
    "NK_CD158e1+",
    "CD4+_T_naive",
    "CD4+_T_activated",
    "CD4+_T_activated_integrinB7+",
    "CD4+_T_CD314+_CD45RA+",
    "CD8+_T_naive",
    "CD8+_T_naive_CD127+_CD26-_CD101-",
    "T_prog_cycling",
    "CD8+_T_CD49f+",
    "CD8+_T_CD69+_CD45RA+",
    "CD8+_T_CD69+_CD45RO+",
    "CD8+_T_TIGIT+_CD45RA+",
    "CD8+_T_TIGIT+_CD45RO+",
    "CD8+_T_CD57+_CD45RA+",
    "CD8+_T_CD57+_CD45RO+",
    "gdT_CD158b+",
    "gdT_TCRVD2+",
    "dnT",
    "T_reg",
    "Naive_CD20+_B_IGKC+",
    "Naive_CD20+_B_IGKC-",
    "Transitional_B",
    "B1_B_IGKC+",
    "B1_B_IGKC-",
    "Plasma_cell_IGKC+",
    "Plasma_cell_IGKC-",
    "Plasmablast_IGKC+",
    "Plasmablast_IGKC-",
    "cDC1",
    "cDC2",
    "pDC",
    "CD14+_Mono",
    "CD16+_Mono",
    "Erythroblast",
    "Normoblast",
    "Proerythroblast",
    "Reticulocyte",
    "MAIT",
]
phenotypes_labels = [
    "HSC",
    "Lymph prog",
    "G/M prog",
    "MK/E prog",
    "ILC",
    "ILC1",
    "NK",
    "NK CD158e1+",
    "CD4+ T naive",
    "CD4+ T activated",
    "CD4+ T activated integrinB7+",
    "CD4+ T CD314+ CD45RA+",
    "CD8+ T naive",
    "CD8+ T naive CD127+ CD26- CD101-",
    "T prog cycling",
    "CD8+ T CD49f+",
    "CD8+ T CD69+ CD45RA+",
    "CD8+ T CD69+ CD45RO+",
    "CD8+ T TIGIT+ CD45RA+",
    "CD8+ T TIGIT+ CD45RO+",
    "CD8+ T CD57+ CD45RA+",
    "CD8+ T CD57+ CD45RO+",
    "gdT CD158b+",
    "gdT TCRVD2+",
    "dnT",
    "T reg",
    "Naive CD20+ B IGKC+",
    "Naive CD20+ B IGKC-",
    "Transitional B",
    "B1 B IGKC+",
    "B1 B IGKC-",
    "Plasma cell IGKC+",
    "Plasma cell IGKC-",
    "Plasmablast IGKC+",
    "Plasmablast IGKC-",
    "cDC1",
    "cDC2",
    "pDC",
    "CD14+ Mono",
    "CD16+ Mono",
    "Erythroblast",
    "Normoblast",
    "Proerythroblast",
    "Reticulocyte",
    "MAIT",
]
phenotypes_dataset = phenotypes_labels


In [ ]:
seeds = ["0"]

graph_csv_go_list = []
graph_csv_index_go_list = []
go_importances = []
for modality in tqdm(["expression"]):
    genes_importances_modality = []
    graph_csv_modality_list = []
    for phenotype, phenotype_label in tqdm(zip(phenotypes, phenotypes_labels)):
        graph_csv_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        graph_csv_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_list)
        graph_csv_modality_list.append(graph_csv)
        genes_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_go_list.append(graph_csv_modality_list)
    graph_csv_index_go_list.append(np.array(graph_csv_modality_list[0].index))
    go_importances.append(genes_importances_modality)


In [ ]:
seeds = ["0"]

graph_csv_go_baseline_list = []
graph_csv_index_go_baseline_list = []
go_baseline_importances = []
for modality in tqdm(["expression"]):
    genes_importances_modality = []
    graph_csv_modality_list = []
    for phenotype, phenotype_label in tqdm(zip(phenotypes, phenotypes_labels)):
        graph_csv_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype_label}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        graph_csv_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_list)
        graph_csv_modality_list.append(graph_csv)
        genes_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_go_baseline_list.append(graph_csv_modality_list)
    graph_csv_index_go_baseline_list.append(np.array(graph_csv_modality_list[0].index))
    go_baseline_importances.append(genes_importances_modality)


## Genes

In [ ]:
phenotypes = [
    "HSC",
    "Lymph_prog",
    "G_M_prog",
    "MK_E_prog",
    "ILC",
    "ILC1",
    "NK",
    "NK_CD158e1+",
    "CD4+_T_naive",
    "CD4+_T_activated",
    "CD4+_T_activated_integrinB7+",
    "CD4+_T_CD314+_CD45RA+",
    "CD8+_T_naive",
    "CD8+_T_naive_CD127+_CD26-_CD101-",
    "T_prog_cycling",
    "CD8+_T_CD49f+",
    "CD8+_T_CD69+_CD45RA+",
    "CD8+_T_CD69+_CD45RO+",
    "CD8+_T_TIGIT+_CD45RA+",
    "CD8+_T_TIGIT+_CD45RO+",
    "CD8+_T_CD57+_CD45RA+",
    "CD8+_T_CD57+_CD45RO+",
    "gdT_CD158b+",
    "gdT_TCRVD2+",
    "dnT",
    "T_reg",
    "Naive_CD20+_B_IGKC+",
    "Naive_CD20+_B_IGKC-",
    "Transitional_B",
    "B1_B_IGKC+",
    "B1_B_IGKC-",
    "Plasma_cell_IGKC+",
    "Plasma_cell_IGKC-",
    "Plasmablast_IGKC+",
    "Plasmablast_IGKC-",
    "cDC1",
    "cDC2",
    "pDC",
    "CD14+_Mono",
    "CD16+_Mono",
    "Erythroblast",
    "Normoblast",
    "Proerythroblast",
    "Reticulocyte",
    "MAIT",
]
phenotypes_labels = [
    "HSC",
    "Lymph prog",
    "G/M prog",
    "MK/E prog",
    "ILC",
    "ILC1",
    "NK",
    "NK CD158e1+",
    "CD4+ T naive",
    "CD4+ T activated",
    "CD4+ T activated integrinB7+",
    "CD4+ T CD314+ CD45RA+",
    "CD8+ T naive",
    "CD8+ T naive CD127+ CD26- CD101-",
    "T prog cycling",
    "CD8+ T CD49f+",
    "CD8+ T CD69+ CD45RA+",
    "CD8+ T CD69+ CD45RO+",
    "CD8+ T TIGIT+ CD45RA+",
    "CD8+ T TIGIT+ CD45RO+",
    "CD8+ T CD57+ CD45RA+",
    "CD8+ T CD57+ CD45RO+",
    "gdT CD158b+",
    "gdT TCRVD2+",
    "dnT",
    "T reg",
    "Naive CD20+ B IGKC+",
    "Naive CD20+ B IGKC-",
    "Transitional B",
    "B1 B IGKC+",
    "B1 B IGKC-",
    "Plasma cell IGKC+",
    "Plasma cell IGKC-",
    "Plasmablast IGKC+",
    "Plasmablast IGKC-",
    "cDC1",
    "cDC2",
    "pDC",
    "CD14+ Mono",
    "CD16+ Mono",
    "Erythroblast",
    "Normoblast",
    "Proerythroblast",
    "Reticulocyte",
    "MAIT",
]
phenotypes_dataset = phenotypes_labels


In [ ]:
seeds = ["0"]

graph_csv_genes_list = []
graph_csv_index_genes_list = []
genes_importances = []
for modality in tqdm(["expression"]):
    genes_importances_modality = []
    graph_csv_modality_list = []
    for phenotype, phenotype_label in tqdm(zip(phenotypes, phenotypes_labels)):
        graph_csv_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_genes_cell_type/{modality}_cell_type_vs_rest_{phenotype_label}_res_unp_act_mask_False_flip_False_genes_evaluated_graph_csv.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        graph_csv_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_list)
        graph_csv_modality_list.append(graph_csv)
        genes_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_genes_list.append(graph_csv_modality_list)
    graph_csv_index_genes_list.append(np.array(graph_csv_modality_list[0].index))
    genes_importances.append(genes_importances_modality)


In [ ]:
seeds = ["0"]

graph_csv_genes_baseline_list = []
graph_csv_index_genes_baseline_list = []
genes_baseline_importances = []
for modality in tqdm(["expression"]):
    genes_importances_modality = []
    graph_csv_modality_list = []
    for phenotype, phenotype_label in tqdm(zip(phenotypes, phenotypes_labels)):
        graph_csv_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_genes_cell_type/{modality}_cell_type_vs_rest_{phenotype_label}_res_unp_act_mask_False_flip_False_genes_evaluated_graph_csv.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        graph_csv_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_list)
        graph_csv_modality_list.append(graph_csv)
        genes_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_genes_baseline_list.append(graph_csv_modality_list)
    graph_csv_index_genes_baseline_list.append(np.array(graph_csv_modality_list[0].index))
    genes_baseline_importances.append(genes_importances_modality)


### Analysis

#### Change in GO importance -> Graph

In [ ]:
all_percentage_changes = []
for phenotype, gene_group, gene_group_baseline in zip(
    phenotypes, go_importances[0], go_baseline_importances[0]
):
    diff = gene_group - gene_group_baseline
    print(phenotype)
    print(np.mean(diff))
    percentage_change = gene_group / gene_group_baseline
    print((np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100)
    all_percentage_changes.append(
        (np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100
    )
    print(np.max(diff))
    print(np.min(diff))
    print("%%%")


In [ ]:
all_percentage_changes = []
for phenotype, gene_group, gene_group_baseline in zip(
    phenotypes, go_importances[0], go_baseline_importances[0]
):
    diff = gene_group - gene_group_baseline
    print(phenotype)
    print(np.mean(diff))
    percentage_change = gene_group / gene_group_baseline
    print((np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100)
    all_percentage_changes.append(
        (np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100
    )
    print(np.max(diff))
    print(np.min(diff))
    print("%%%")


In [ ]:
index_phenotype = 38
diff = go_importances[0][index_phenotype] - go_baseline_importances[0][index_phenotype]
print(phenotypes[index_phenotype])
print(np.mean(diff))
percentage_change = go_importances[0][index_phenotype] - go_baseline_importances[0][index_phenotype]
print((np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100)
print(np.max(diff))
print(np.min(diff))
print(percentage_change[percentage_change > np.quantile(percentage_change, 0.995)])
print(
    np.array(graph_csv_index_go_list[0])[percentage_change > np.quantile(percentage_change, 0.995)]
)
print(
    [
        obodag[go_term].name
        for go_term in np.array(graph_csv_index_go_list[0])[
            percentage_change > np.quantile(percentage_change, 0.995)
        ]
    ]
)
print("%%%")


In [ ]:
index_phenotype = 39
diff = go_importances[0][index_phenotype] - go_baseline_importances[0][index_phenotype]
print(phenotypes[index_phenotype])
print(np.mean(diff))
percentage_change = go_importances[0][index_phenotype] - go_baseline_importances[0][index_phenotype]
print((np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100)
print(np.max(diff))
print(np.min(diff))
print(percentage_change[percentage_change > np.quantile(percentage_change, 0.995)])
print(
    np.array(graph_csv_index_go_list[0])[percentage_change > np.quantile(percentage_change, 0.995)]
)
print(go_importances[0][index_phenotype][percentage_change > np.quantile(percentage_change, 0.995)])
print(
    go_baseline_importances[0][index_phenotype][
        percentage_change > np.quantile(percentage_change, 0.995)
    ]
)
print(
    [
        obodag[go_term].name
        for go_term in np.array(graph_csv_index_go_list[0])[
            percentage_change > np.quantile(percentage_change, 0.995)
        ]
    ]
)
print("%%%")


In [ ]:
from scipy.stats import bootstrap, wilcoxon

all_diffs_flat = []
for phenotype, gene_group, gene_group_baseline in zip(
    phenotypes, go_importances[0], go_baseline_importances[0]
):
    ratio = np.mean(gene_group) / np.mean(gene_group_baseline)
    all_diffs_flat.append(ratio)

all_diffs_flat = np.array(all_diffs_flat)
res = bootstrap((all_diffs_flat,), np.mean, n_resamples=10000, confidence_level=0.95)
print(f"Mean ratio: {np.mean(all_diffs_flat):.4f}")
print(f"95% CI: [{res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f}]")

baseline_means = [np.mean(g) for g in go_baseline_importances[0]]
model_means = [np.mean(g) for g in go_importances[0]]
stat, p = wilcoxon(model_means, baseline_means)
print(f"Wilcoxon signed-rank: stat={stat:.2f}, p={p:.4e}")


In [ ]:
all_abs_diffs = []
for phenotype, gene_group, gene_group_baseline in zip(
    phenotypes, go_importances[0], go_baseline_importances[0]
):
    all_abs_diffs.append(np.mean(gene_group) - np.mean(gene_group_baseline))

all_abs_diffs = np.array(all_abs_diffs)
res_abs = bootstrap((all_abs_diffs,), np.mean, n_resamples=10000, confidence_level=0.95)
stat, p = wilcoxon(
    [np.mean(g) for g in go_importances[0]], [np.mean(g) for g in go_baseline_importances[0]]
)

print(f"Mean ΔAUROC: {np.mean(all_abs_diffs):.4f}")
print(f"95% CI: [{res_abs.confidence_interval.low:.4f}, {res_abs.confidence_interval.high:.4f}]")
print(f"Wilcoxon p = {p:.4e}")


In [ ]:
all_percentage_changes_boot = []
for phenotype, gene_group, gene_group_baseline in zip(
    phenotypes, go_importances[0], go_baseline_importances[0]
):
    mask = gene_group_baseline > 0
    ratio_per_term = gene_group[mask] / gene_group_baseline[mask]
    all_percentage_changes_boot.append(np.mean(ratio_per_term))

all_percentage_changes_boot = np.array(all_percentage_changes_boot)

res = bootstrap((all_percentage_changes_boot,), np.mean, n_resamples=10000, confidence_level=0.95)
stat, p = wilcoxon(
    all_percentage_changes_boot, np.ones_like(all_percentage_changes_boot)
)

print(
    f"Mean ratio: {np.mean(all_percentage_changes_boot):.4f}  "
    f"({(np.mean(all_percentage_changes_boot)-1)*100:.1f}%)"
)
print(
    f"95% CI: [{res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f}]  "
    f"({(res.confidence_interval.low-1)*100:.1f}% – {(res.confidence_interval.high-1)*100:.1f}%)"
)
print(f"Wilcoxon vs ratio=1: stat={stat:.1f}, p={p:.4e}")


In [ ]:
all_percentage_changes_boot = []
for phenotype, gene_group, gene_group_baseline in zip(
    phenotypes, go_importances[0], go_baseline_importances[0]
):
    mask = gene_group_baseline > 0.51
    ratio_per_term = gene_group[mask] / gene_group_baseline[mask]
    all_percentage_changes_boot.append(np.mean(ratio_per_term))

all_percentage_changes_boot = np.array(all_percentage_changes_boot)

res = bootstrap((all_percentage_changes_boot,), np.mean, n_resamples=10000, confidence_level=0.95)
stat, p = wilcoxon(
    all_percentage_changes_boot, np.ones_like(all_percentage_changes_boot)
)

print(
    f"Mean ratio: {np.mean(all_percentage_changes_boot):.4f}  "
    f"({(np.mean(all_percentage_changes_boot)-1)*100:.1f}%)"
)
print(
    f"95% CI: [{res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f}]  "
    f"({(res.confidence_interval.low-1)*100:.1f}% – {(res.confidence_interval.high-1)*100:.1f}%)"
)
print(f"Wilcoxon vs ratio=1: stat={stat:.1f}, p={p:.4e}")


In [ ]:
gene_group_cd14_mono = go_importances[0][np.where(np.array(phenotypes) == "CD14+_Mono")[0][0]]
gene_group_baseline_cd14_mono = go_baseline_importances[0][
    np.where(np.array(phenotypes) == "CD14+_Mono")[0][0]
]


In [ ]:
for ct in ["CD14+_Mono", "CD16+_Mono"]:
    idx = np.where(np.array(phenotypes) == ct)[0][0]
    model_vals = go_importances[0][idx]
    base_vals = go_baseline_importances[0][idx]
    ratios = model_vals / base_vals
    res = bootstrap((ratios,), np.mean, n_resamples=10000, confidence_level=0.95)
    stat, p = wilcoxon(model_vals, base_vals)
    print(
        f"{ct}: mean ratio={np.mean(ratios):.4f}, 95% CI=[{res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f}], Wilcoxon p={p:.4e}"
    )


In [ ]:
go_diff_cd14_mono = gene_group_cd14_mono - gene_group_baseline_cd14_mono
go_diff_cd14_mono


In [ ]:
from collections import Counter

Counter(
    [
        obodag[go_term].level
        for go_term in graph_csv_index_go_list[0][
            np.where(go_diff_cd14_mono < np.quantile(go_diff_cd14_mono, 0.05))
        ]
    ]
)


In [ ]:
Counter(
    [obodag[go_term].level for go_term in graph_csv_index_go_list[0] if go_term in obodag.keys()]
)


In [ ]:
np.mean(go_diff_cd14_mono)


In [ ]:
for go_term_index, go_term in enumerate(graph_csv_index_go_list[0]):
    break


In [ ]:
go_term = "GO:0042048"


In [ ]:
level1_terms = [
    go_term
    for go_term in obodag.keys()
    if obodag[go_term].level == 1 and obodag[go_term].namespace == "biological_process"
]


In [ ]:
for level1_term in level1_terms:
    try:
        all_terms = go_diff_cd14_mono[
            [
                go_term_index
                for go_term_index, go_term in enumerate(graph_csv_index_go_list[0])
                if go_term in obodag.keys() and level1_term in obodag[go_term].get_all_parents()
            ]
        ]
        print("%%%")
        print(level1_term)
        print(obodag[level1_term].name)
        print(len(all_terms))
        print(Counter(all_terms > 0))
        print(Counter(all_terms > 0)[True] / len(all_terms))
        print(np.mean(all_terms))
        print(np.max(all_terms))
        print(np.min(all_terms))
    except:
        continue


In [ ]:
np.mean(
    go_diff_cd14_mono[
        [
            go_term_index
            for go_term_index, go_term in enumerate(graph_csv_index_go_list[0])
            if go_term in obodag.keys() and obodag[go_term].level == 1
        ]
    ]
)


In [ ]:
np.mean(
    go_diff_cd14_mono[
        [
            go_term_index
            for go_term_index, go_term in enumerate(graph_csv_index_go_list[0])
            if go_term in obodag.keys() and obodag[go_term].level == 2
        ]
    ]
)


In [ ]:
np.mean(
    go_diff_cd14_mono[
        [
            go_term_index
            for go_term_index, go_term in enumerate(graph_csv_index_go_list[0])
            if go_term in obodag.keys() and obodag[go_term].level == 3
        ]
    ]
)


In [ ]:
np.mean(
    go_diff_cd14_mono[
        [
            go_term_index
            for go_term_index, go_term in enumerate(graph_csv_index_go_list[0])
            if go_term in obodag.keys() and obodag[go_term].level == 4
        ]
    ]
)


In [ ]:
np.mean(
    go_diff_cd14_mono[
        [
            go_term_index
            for go_term_index, go_term in enumerate(graph_csv_index_go_list[0])
            if go_term in obodag.keys() and obodag[go_term].level == 5
        ]
    ]
)


In [ ]:
go_diff_cd14_mono = gene_group_cd14_mono - gene_group_baseline_cd14_mono
set_figure_params('single', 2.2)
fig, ax = plt.subplots()
ax.hist(go_diff_cd14_mono, bins=40, color='#70acc0', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='#c12075', linewidth=0.8, linestyle='--')
n_pos = np.sum(go_diff_cd14_mono > 0)
n_neg = np.sum(go_diff_cd14_mono < 0)
ax.set_xlabel('GO importance change (model − baseline)')
ax.set_ylabel('Number of GO terms')
ax.set_title(f'CD14+ Monocytes: {n_pos} terms positive, {n_neg} terms negative')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, 'go_diff_distribution_cd14.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, 'go_diff_distribution_cd14.svg'), bbox_inches='tight', format='svg')
plt.show()


In [ ]:
plt.hist(go_diff_cd14_mono)
plt.show()


In [ ]:
from matplotlib.colors import Normalize, to_hex
from matplotlib.cm import ScalarMappable

norm = Normalize(vmin=-0.5, vmax=0.5)
mappable = ScalarMappable(norm=norm, cmap=cmap_heat)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 1))
fig.subplots_adjust(bottom=0.5)

cbar = plt.colorbar(mappable, cax=ax, orientation="horizontal")
cbar.set_ticks([-0.5, 0.0, 0.5])
cbar.set_ticklabels(["-0.5", "0.0", "0.5"])
cbar.set_label("Value")

plt.show()


In [ ]:
go_term = "GO:0008150"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0002376"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0002252"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0002443"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0001909"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0002449"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0002444"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0002520"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0048534"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0002200"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0002507"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0050896"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0051775"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0009628"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0014823"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0043500"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0009987"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0008283"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0070661"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0032943"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0061517"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0070662"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


In [ ]:
go_term = "GO:0002158"
print(go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0])
print(
    to_hex(
        mappable.to_rgba(
            go_diff_cd14_mono[np.where(graph_csv_index_go_baseline_list[0] == go_term)][0]
        )
    )
)


#### Change in gene importances

In [ ]:
all_percentage_changes = []
for phenotype, gene_group, gene_group_baseline in zip(
    phenotypes, genes_importances[0], genes_baseline_importances[0]
):
    diff = gene_group - gene_group_baseline
    print(phenotype)
    print(np.mean(diff))
    percentage_change = gene_group / gene_group_baseline
    print((np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100)
    all_percentage_changes.append(
        (np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100
    )
    print(np.max(diff))
    print(np.min(diff))
    print("%%%")


In [ ]:
np.mean(all_percentage_changes)


# Change GO/ TAD further ablations

In [ ]:
gene_importances_further_comp = []
for phenotype_label in tqdm(
    ["CD14+ Mono", "CD16+ Mono", "Naive CD20+ B IGKC+", "NK", "pDC", "CD4+ T naive", "CD8+ T naive"]
):
    graph_csv = pd.read_pickle(
        f"../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_genes_cell_type/expression_cell_type_vs_rest_{phenotype_label}_res_unp_act_mask_False_flip_False_genes_evaluated_graph_csv.pkl"
    )
    graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
    graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
    gene_importances_further_comp.append(np.array(graph_csv["predictors_label_valid_roc"]))


In [ ]:
gene_groups_importances_further_comp = []
for phenotype_label in tqdm(
    ["CD14+ Mono", "CD16+ Mono", "Naive CD20+ B IGKC+", "NK", "pDC", "CD4+ T naive", "CD8+ T naive"]
):
    graph_csv = pd.read_pickle(
        f"../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/expression_cell_type_vs_rest_{phenotype_label.replace(' ', '_')}_res_unp_act_mask_False_flip_False_gene_groups_ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_gene_groups_evaluated_graph_csv.pkl"
    )
    graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
    graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
    gene_groups_importances_further_comp.append(np.array(graph_csv["predictors_label_valid_roc"]))


In [ ]:
for comp in ["ENCFF164AGX", "ENCFF203AKP", "ENCFF804SET", "ENCFF173VDJ_shifted_benchmark"]:
    print(comp)
    for phenotype_index, phenotype_label in tqdm(
        enumerate(
            [
                "CD14+ Mono",
                "CD16+ Mono",
                "Naive CD20+ B IGKC+",
                "NK",
                "pDC",
                "CD4+ T naive",
                "CD8+ T naive",
            ]
        )
    ):
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_{comp}_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_{phenotype_label}_res_unp_act_mask_False_flip_False_genes_evaluated_graph_csv_with_ci.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        gene_group = np.array(graph_csv["predictors_label_valid_roc"]).astype("float")
        gene_group_baseline = gene_importances_further_comp[phenotype_index].astype("float")
        diff = gene_group - gene_group_baseline
        print(phenotype_label)
        print(np.mean(diff))
        percentage_change = gene_group / gene_group_baseline
        print((np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100)
        print(np.max(diff))
        print(np.min(diff))
        print("%%%")


In [ ]:
for comp in ["ENCFF164AGX", "ENCFF203AKP", "ENCFF804SET", "ENCFF173VDJ_shifted_benchmark"]:
    print(comp)
    for phenotype_index, phenotype_label in tqdm(
        enumerate(
            [
                "CD14+ Mono",
                "CD16+ Mono",
                "Naive CD20+ B IGKC+",
                "NK",
                "pDC",
                "CD4+ T naive",
                "CD8+ T naive",
            ]
        )
    ):
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_{comp}_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_rest_{phenotype_label}_res_unp_act_mask_False_flip_False_gene_groups_ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_gene_groups_evaluated_graph_csv_with_ci.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        gene_group = np.array(graph_csv["predictors_label_valid_roc"]).astype("float")
        gene_group_baseline = gene_groups_importances_further_comp[phenotype_index].astype("float")
        diff = gene_group - gene_group_baseline
        print(phenotype_label)
        print(np.mean(diff))
        percentage_change = gene_group / gene_group_baseline
        print((np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100)
        print(np.max(diff))
        print(np.min(diff))
        print("%%%")


#### Change in TAD importances

In [ ]:
all_percentage_changes = []
for phenotype, gene_group, gene_group_baseline in zip(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))],
    gene_groups_importances[0],
    gene_groups_baseline_importances[0],
):
    diff = gene_group - gene_group_baseline
    print(phenotype)
    print(np.mean(diff))
    percentage_change = gene_group / gene_group_baseline
    print((np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100)
    all_percentage_changes.append(
        (np.mean(percentage_change[np.isfinite(percentage_change)]) - 1) * 100
    )
    print(np.max(diff))
    print(np.min(diff))
    print("%%%")


In [ ]:
all_true_changes = []
for phenotype, gene_group, gene_group_baseline in zip(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))],
    gene_groups_importances[0],
    gene_groups_baseline_importances[0],
):
    diff = gene_group - gene_group_baseline
    print(phenotype)
    percentage_change = gene_group / gene_group_baseline
    all_true_changes.append(
        Counter(percentage_change[np.isfinite(percentage_change)] >= 1)[True]
        / len(np.isfinite(percentage_change))
    )
    print(
        Counter(percentage_change[np.isfinite(percentage_change)] >= 1)[True]
        / len(np.isfinite(percentage_change))
    )
    print(
        Counter(percentage_change[np.isfinite(percentage_change)] >= 1)[False]
        / len(np.isfinite(percentage_change))
    )


In [ ]:
np.mean(all_true_changes)


In [ ]:
np.mean(all_percentage_changes)


In [ ]:
from scipy.stats import bootstrap, wilcoxon

all_gene_diffs = []
for phenotype, gene_group, gene_group_baseline in zip(
    phenotypes, genes_importances[0], genes_baseline_importances[0]
):
    ratio = np.mean(gene_group) / np.mean(gene_group_baseline)
    all_gene_diffs.append(ratio)

all_gene_diffs = np.array(all_gene_diffs)
res_gene = bootstrap((all_gene_diffs,), np.mean, n_resamples=10000, confidence_level=0.95)
print(
    f"Gene importance mean ratio: {np.mean(all_gene_diffs):.4f}, 95% CI=[{res_gene.confidence_interval.low:.4f}, {res_gene.confidence_interval.high:.4f}]"
)

all_tad_diffs = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    tad_ratio = np.mean(graph_csv_list[0][phenotype_index]["predictors_label_valid_roc"]) / np.mean(
        graph_csv_baseline_list[0][phenotype_index]["predictors_label_valid_roc"]
    )
    all_tad_diffs.append(tad_ratio)

all_tad_diffs = np.array(all_tad_diffs)
res_tad = bootstrap((all_tad_diffs,), np.mean, n_resamples=10000, confidence_level=0.95)
print(
    f"TAD importance mean ratio: {np.mean(all_tad_diffs):.4f}, 95% CI=[{res_tad.confidence_interval.low:.4f}, {res_tad.confidence_interval.high:.4f}]"
)


#### Top TAD change groups

In [ ]:
tads_of_interest = [tad for tad in graph_csv_list[0][0].index if len(tad.split("_")) > 2]


In [ ]:
all_importances = []
for tad_of_interest in tads_of_interest:
    tad_importances = []
    for phenotype_index, phenotype in enumerate(
        [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
    ):
        tad_importances.append(
            graph_csv_list[0][phenotype_index].loc[tad_of_interest]["predictors_label_valid_roc"]
        )
    all_importances.append(tad_importances)


In [ ]:
indices_selected = set()
diffs = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    diff = (
        graph_csv_list[0][phenotype_index].loc[tads_of_interest]["predictors_label_valid_roc"]
        - graph_csv_baseline_list[0][phenotype_index].loc[tads_of_interest][
            "predictors_label_valid_roc"
        ]
    )
    diffs.append(np.array(diff))
indices_selected


In [ ]:
indices_selected = set()
diffs = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    diff = (
        graph_csv_list[0][phenotype_index]["predictors_label_valid_roc"]
        - graph_csv_baseline_list[0][phenotype_index]["predictors_label_valid_roc"]
    )
    diffs.append(np.array(diff))
indices_selected


In [ ]:
diffs = np.array(diffs)


In [ ]:
indices_selected = np.array(
    [
        list(graph_csv_list[0][0].index).index("ENSG00000146192_ENSG00000164530_ENSG00000198663"),
        list(graph_csv_list[0][0].index).index("ENSG00000164307_ENSG00000247121_ENSG00000248734"),
    ]
)


In [ ]:
names_indices_selected = [
    "TAD: FGD2, PI16, C6orf89",
    "TAD: ERAP1, ENSG00000247121 (lncRNA, antisense to LNPEP and ERAP2), ENSG00000248734 (lncRNA)",
]


In [ ]:
mdata_counts.mod["rna"].var.index = mdata_counts.mod["rna"].var["gene_stable_id"]


In [ ]:
phenotypes_labels_tad_groups = [
    "CD14+ Mono",
    "CD16+ Mono",
    "CD4+ T naive",
    "CD8+ T naive",
    "NK",
    "Normoblast",
    "Erythroblast",
]


In [ ]:
indices_phenotypes_labels_tad_groups = [
    sorted(phenotypes_labels).index(phenotype) for phenotype in phenotypes_labels_tad_groups
]


In [ ]:
import scanpy as sc

for genes in [
    ["ENSG00000231500", "ENSG00000235863", "ENSG00000237441"],
    ["ENSG00000170180", "ENSG00000197465", "ENSG00000250361"],
    ["ENSG00000164307", "ENSG00000247121", "ENSG00000248734"],
    ["ENSG00000112640", "ENSG00000112659", "ENSG00000124574", "ENSG00000171467", "ENSG00000171611"],
    ["ENSG00000146192", "ENSG00000164530", "ENSG00000198663"],
]:
    sc.pl.dotplot(
        mdata_counts.mod["rna"][
            mdata_counts.mod["rna"].obs[labels_key].isin(phenotypes_labels_tad_groups)
        ],
        genes,
        groupby=labels_key,
        categories_order=phenotypes_labels_tad_groups,
    )


In [ ]:
for phenotype in np.unique(mdata_counts.mod["rna"].obs[labels_key]):
    obs_indices = np.where(mdata_counts.mod["rna"].obs[labels_key] == phenotype)[0]
    for gene in ["ENSG00000164307", "ENSG00000247121", "ENSG00000248734"]:
        var_index = np.where(mdata_counts.mod["rna"].var["gene_stable_id"] == gene)[0][0]
        plt.hist(mdata_counts.mod["rna"].X[obs_indices, var_index])
        plt.title(f"{phenotype} - {gene}")
        plt.show()
        plt.close()


In [ ]:
import scanpy as sc

for genes in [
    [
        "ENSG00000146192",
        "ENSG00000164530",
        "ENSG00000198663",
        "ENSG00000164307",
        "ENSG00000247121",
        "ENSG00000248734",
    ]
]:
    _gene_symbols = {
        "ENSG00000146192": "FGD2",
        "ENSG00000164530": "PI16",
        "ENSG00000198663": "C6orf89",
        "ENSG00000164307": "ERAP1",
    }
    _dp = sc.pl.dotplot(
        mdata_counts.mod["rna"][
            mdata_counts.mod["rna"].obs[labels_key].isin(phenotypes_labels_tad_groups)
        ],
        genes,
        groupby=labels_key,
        categories_order=phenotypes_labels_tad_groups,
        show=False,
        return_fig=True,
    )
    _dp.make_figure()
    _dp.ax_dict["mainplot_ax"].set_xticklabels(
        [_gene_symbols.get(g, g) for g in genes]
    )
    plt.savefig(
        os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_tad_case_study_genes_dotplot.png"),
        dpi=500, format="png", transparent=True, bbox_inches="tight",
    )
    plt.savefig(
        os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_tad_case_study_genes_dotplot.svg"),
        dpi=500, transparent=True, bbox_inches="tight", format="svg",
    )
    plt.show()


In [ ]:
df = pd.DataFrame(
    np.array(diffs)[:, list(indices_selected)][indices_phenotypes_labels_tad_groups],
    columns=names_indices_selected,
    index=phenotypes_labels_tad_groups,
)


In [ ]:
_g = sns.clustermap(df, cmap=cmap, figsize=(3, 2.8), row_cluster=False, col_cluster=False)
plt.tight_layout()
_g.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_tad_importance_change_case_study_heatmap.png"),
    dpi=500, format="png", transparent=True, bbox_inches="tight",
)
_g.savefig(
    os.path.join(_OUTDIR_ABS, "neurips2021_cite_bmmc_tad_importance_change_case_study_heatmap.svg"),
    dpi=500, transparent=True, bbox_inches="tight", format="svg",
)
plt.show()


#### Annotation of selected TAD groups

In [ ]:
genes_of_interest_groups = [
    ["ENSG00000146192", "ENSG00000164530", "ENSG00000198663"],
    ["ENSG00000164307", "ENSG00000247121", "ENSG00000248734"],
]


In [ ]:
for phenotype in phenotypes_labels_tad_groups:
    break
    phenotype = sorted(phenotypes_labels)[index]
    break


In [ ]:
for genes_of_interest_group in genes_of_interest_groups:
    print(genes_of_interest_group)
    for phenotype in phenotypes_labels_tad_groups:
        print(phenotype)
        for gene_of_interest in genes_of_interest_group:
            print(gene_of_interest)
            diff = (
                graph_csv_genes_list[0][phenotypes_labels.index(phenotype)].loc[gene_of_interest][
                    "predictors_label_valid_roc"
                ]
                - graph_csv_genes_baseline_list[0][phenotypes_labels.index(phenotype)].loc[
                    gene_of_interest
                ]["predictors_label_valid_roc"]
            )
            print(diff)
        print("%%%")


In [ ]:
phenotypes_labels.index(phenotype)


In [ ]:
for phenotype, phenotype_label in tqdm(zip(phenotypes, phenotypes_labels)):
    genes_importances.append(genes_importances_modality)
    graph_csv_genes_list.append(graph_csv_modality_list)

    graph_csv_genes_baseline_list.append(graph_csv_modality_list)
    graph_csv_index_genes_baseline_list.append(np.array(graph_csv_modality_list[0].index))
    genes_baseline_importances.append(genes_importances_modality)


#### How many additional genes now have an annotation due to TADs?

In [ ]:
modality = "expression"
phenotype = phenotypes[0]
go_csv = pd.read_pickle(
    "../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/expression_cell_type_vs_Erythroid_cell_T_reg_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
)
genes_csv = pd.read_pickle(
    f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_gene_groups_ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_gene_groups_evaluated_graph_csv.pkl"
)


In [ ]:
import pickle

with open(
    "../.experiment_data/primary/_biological_process_go-basic_goa_human_ensembl_gene_mapping_2024-08-08_goa_human_isoform_ensembl_gene_mapping_2024-08-08_goa_human_rna_ensembl_gene_mapping_2024-08-08.v2_processed.pkl",
    "rb",
) as f:
    ensembl2go = pickle.load(f)


In [ ]:
for gene in [
    "ENSG00000101665",
    "ENSG00000134030",
    "ENSG00000196466",
    "ENSG00000196646",
    "ENSG00000198799",
    "ENSG00000226419",
    "ENSG00000263069",
    "ENSG00000262580",
    "ENSG00000275713",
    "ENSG00000274750",
]:
    print(gene)
    print(gene in ensembl2go.keys())


In [ ]:
for gene in [
    "ENSG00000231500",
    "ENSG00000235863",
    "ENSG00000237441",
    "ENSG00000170180",
    "ENSG00000197465",
    "ENSG00000250361",
    "ENSG00000164307",
    "ENSG00000247121",
    "ENSG00000248734",
    "ENSG00000112640",
    "ENSG00000112659",
    "ENSG00000124574",
    "ENSG00000171467",
    "ENSG00000171611",
    "ENSG00000146192",
    "ENSG00000164530",
    "ENSG00000198663",
]:
    print(gene)
    print(gene in ensembl2go.keys())


In [ ]:
tads_conveying_additional_information = 0
genes_mapped = []
for tad in genes_csv.index:
    genes = tad.split("_")
    genes_not_in_ensembl2go = [gene for gene in genes if not gene in ensembl2go.keys()]
    if len(genes_not_in_ensembl2go) > 0:
        tads_conveying_additional_information += 1
        genes_mapped += genes_not_in_ensembl2go


#### Correlation TAD improvement/ importance with Hi-C scores

In [ ]:
phenotype_index = 0


In [ ]:
gene_groups = np.load(
    "../.experiment_data/primary/preprocessing/resources/contact_domains/encode/ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_gene_groups.npy",
    allow_pickle=True,
)
gene_groups


In [ ]:
hi_c_scores = np.load(
    "../.experiment_data/primary/preprocessing/resources/contact_domains/encode/ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_score_gene_groups.npy"
)
hi_c_scores


In [ ]:
phenotype_index = 0


In [ ]:
index_gene_groups = []
for gene_group in tqdm(graph_csv_index_list[phenotype_index]):
    gene_group = gene_group.split("_")
    for hi_c_gene_group_index, hi_c_gene_group in tqdm(enumerate(gene_groups)):
        if np.all([gene in hi_c_gene_group for gene in gene_group]):
            index_gene_groups.append(hi_c_gene_group_index)
            break


In [ ]:
for phenotype_index, phenotype in tqdm(
    enumerate([x for _, x in sorted(zip(phenotypes_labels, phenotypes))])
):
    print(phenotype)
    print(
        scipy.stats.spearmanr(
            graph_csv_list[0][phenotype_index]["predictors_label_valid_roc"],
            hi_c_scores[index_gene_groups],
        )
    )
    print(
        scipy.stats.spearmanr(
            graph_csv_list[0][phenotype_index]["predictors_label_valid_roc"]
            - graph_csv_baseline_list[0][phenotype_index]["predictors_label_valid_roc"],
            hi_c_scores[index_gene_groups],
        )
    )


In [ ]:
all_interactions = np.load("../.experiment_data/primary/all_interactions.npy")
all_interactions_comb_translated = np.load(
    "../.experiment_data/primary/all_interactions_comb_translated.npy"
)


In [ ]:
np.max(all_interactions)


In [ ]:
np.quantile(all_interactions, 0.999)


In [ ]:
np.where(all_interactions > np.quantile(all_interactions, 0.999))


In [ ]:
import pandas as pd

_GENE_MAPPING_PATH = os.path.abspath(
    os.path.join(
        "..", "preprocessing", "resources", "ensembl_gene_id_gene_name_mapping",
        "ensembl_gene_id_gene_name_mapping.csv",
    )
)

def _get_ensembl_gene_id_gene_name_mapping():
    return pd.read_csv(_GENE_MAPPING_PATH)


In [ ]:
ensembl_gene_id_gene_name_mapping = _get_ensembl_gene_id_gene_name_mapping()


In [ ]:
count = -1
for gene in all_interactions_comb_translated[
    np.where(all_interactions > np.quantile(all_interactions, 0.995))
].flatten():
    count += 1
    print(
        ensembl_gene_id_gene_name_mapping[
            ensembl_gene_id_gene_name_mapping["Gene stable ID"] == gene
        ]["Gene name"].iloc[0]
    )
    if count % 2:
        print("%%%")


In [ ]:
all_interactions_comb_translated = np.sort(all_interactions_comb_translated, axis=1)


In [ ]:
phenotype_index = 0


In [ ]:
import itertools


In [ ]:
all_interactions_comb_translated = np.squeeze(all_interactions_comb_translated)


In [ ]:
all_interactions_gene_groups = []
for gene_group in tqdm(graph_csv_index_list[phenotype_index]):
    gene_group = gene_group.split("_")
    all_interactions_gene_group = []
    for gene_group_comb in itertools.combinations(gene_group, 2):
        all_interactions_gene_group.append(
            all_interactions[
                np.where(
                    np.all(all_interactions_comb_translated == np.sort(gene_group_comb), axis=1)
                )[0]
            ]
        )
    all_interactions_gene_groups.append(
        np.mean(np.abs(np.concatenate(all_interactions_gene_group)))
    )


In [ ]:
for phenotype_index, phenotype in tqdm(
    enumerate([x for _, x in sorted(zip(phenotypes_labels, phenotypes))])
):
    print(phenotype)
    print(
        scipy.stats.spearmanr(
            graph_csv_list[0][phenotype_index]["predictors_label_valid_roc"],
            all_interactions_gene_groups,
        )
    )
    print(
        scipy.stats.spearmanr(
            graph_csv_list[0][phenotype_index]["predictors_label_valid_roc"]
            - graph_csv_baseline_list[0][phenotype_index]["predictors_label_valid_roc"],
            all_interactions_gene_groups,
        )
    )


#### Correlation with genomic markers

In [ ]:
import pyBigWig

def split_and_average(data, num_groups):
    total_elements = len(data)
    base_size = total_elements // num_groups
    extra = total_elements % num_groups
    means = []
    idx = 0
    for i in range(num_groups):
        group_size = base_size + (1 if i < extra else 0)
        group = data[idx:idx + group_size]
        means.append(np.mean(group))
        idx += group_size
    return means

_IRF_TRACKS_DIR = os.path.abspath(
    os.path.join("..", "preprocessing", "resources", "tad2_irf_tracks")
)
irf_bigwig_paths = {
    "IRF1": os.path.join(_IRF_TRACKS_DIR, "ENCFF630FXX_K562_IRF1_GRCh38_2020-12-01_fc_chr5_96760000_97038000.bigWig"),
    "IRF2": os.path.join(_IRF_TRACKS_DIR, "ENCFF887HOR_K562_IRF2_GRCh38_2020-12-01_fc_chr5_96760000_97038000.bigWig"),
}

for tf_name, path_genomic_marker in irf_bigwig_paths.items():
    bw = pyBigWig.open(path_genomic_marker)
    values = bw.values("chr5", 96760000, 97038000)
    values = split_and_average(values, 745)
    bw.close()

    fig, ax = plt.subplots(figsize=(7.49, 1.5))
    ax.bar(list(range(745)), values)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_ylabel(f"{tf_name} fold change\nover control")
    ax.set_xticks([])
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(os.path.join(_OUTDIR_ABS, f"{tf_name.lower()}_fold_change_tad2.pdf"), bbox_inches="tight")
    plt.savefig(os.path.join(_OUTDIR_ABS, f"{tf_name.lower()}_fold_change_tad2.svg"), bbox_inches="tight", format="svg")
    plt.savefig(os.path.join(_OUTDIR_ABS, f"{tf_name.lower()}_fold_change_tad2.png"), bbox_inches="tight", dpi=500)
    plt.show()


### TAD variance vs. rank

In [ ]:
all_data = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    all_data.append(
        np.array(
            (graph_csv_list[0][phenotype_index] - graph_csv_baseline_list[0][phenotype_index])[
                "predictors_label_valid_roc"
            ]
        )
    )
all_data = np.array(all_data).T


In [ ]:
all_data = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    all_data.append(
        np.array(
            (graph_csv_list[0][phenotype_index] - graph_csv_baseline_list[0][phenotype_index])[
                "predictors_label_valid_roc"
            ]
        )
    )
all_data = np.array(all_data).T


In [ ]:
rho = scipy.stats.spearmanr(np.var(all_data, axis=1), np.mean(all_data, axis=1)).statistic


In [ ]:
rho, p_exact = scipy.stats.spearmanr(np.var(all_data, axis=1), np.mean(all_data, axis=1))
print(f"Spearman rho = {rho:.3f}, p = {p_exact:.4e}")
rho_boots = []
for _ in range(10000):
    idx = np.random.choice(len(all_data), len(all_data), replace=True)
    rho_boots.append(
        scipy.stats.spearmanr(
            np.var(all_data[idx], axis=1), np.mean(all_data[idx], axis=1)
        ).statistic
    )
print(
    f"Bootstrap 95% CI on rho: [{np.percentile(rho_boots, 2.5):.3f}, {np.percentile(rho_boots, 97.5):.3f}]"
)


In [ ]:
from adjustText import adjust_text
set_figure_params('single', 2.5)
plt.figure()
plt.scatter(np.var(all_data, axis=1), np.mean(all_data, axis=1), alpha=0.5, s=3, label=f'Spearman r = {rho:.2f}, p<0.05')
plt.xlabel('Variance of TAD importance across cell types')
plt.ylabel('Mean TAD importance change across cell types')
plt.legend(loc='upper center', bbox_to_anchor=(0.8, 1.1), ncol=1)
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, 'corr_var_cell_type_tad_mean.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, 'corr_var_cell_type_tad_mean.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


In [ ]:
from adjustText import adjust_text

set_figure_params("single", 2.5)
plt.figure()

plt.scatter(
    np.var(all_data, axis=1),
    np.mean(all_data, axis=1),
    alpha=0.5,
    s=3,
    label=f"Spearman r = {rho:.2f}, p<0.05",
)
plt.xlabel("Variance of TAD importance across cell types")
plt.ylabel("Mean TAD importance change across cell types")
plt.legend(loc="upper center", bbox_to_anchor=(0.8, 1.1), ncol=1)

important_idx = [250, 190]
gene_names = ["TAD 1: FGD2, PI16, C6orf89", "TAD2: ERAP1, ENSG00000247121, ENSG00000248734"]

texts = []
for idx, name in zip(important_idx, gene_names):
    texts.append(
        plt.text(np.var(all_data, axis=1)[idx], np.mean(all_data, axis=1)[idx], name, fontsize=8)
    )

adjust_text(texts, arrowprops=dict(arrowstyle="->", color="gray", lw=0.5))
plt.tight_layout()
plt.show()


In [ ]:
set_figure_params("single", 2.5)
plt.scatter(np.var(all_data, axis=1), np.mean(all_data, axis=1), alpha=0.5, s=3)
plt.xlabel("Variation across cell types")
plt.ylabel("Mean TAD importance change across all cell types")
plt.legend(f"TEST")
plt.show()


In [ ]:
all_data = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    all_data.append(np.array((graph_csv_list[0][phenotype_index])["predictors_label_valid_roc"]))
all_data = np.array(all_data).T


In [ ]:
from scipy.stats import f_oneway

cell_type_importances = [all_data[:, i] for i in range(all_data.shape[1])]

F_statistic, p_value = f_oneway(*cell_type_importances)

print("F-statistic:", F_statistic)
print("p-value:", p_value)


In [ ]:
from scipy.stats import f_oneway

cell_type_importances = [all_data[:, i] for i in range(all_data.shape[1])]

F_statistic, p_value = f_oneway(*cell_type_importances)

print("F-statistic:", F_statistic)
print("p-value:", p_value)


### TADs with cell type/ state markers?

In [ ]:
panglaodb = pd.read_csv(
    "../.experiment_data/primary/PanglaoDB_markers_27_Mar_2020.tsv", delimiter="\t"
)
panglaodb


In [ ]:
marker_ensembl_gene_ids = []
for gene in np.unique(panglaodb[panglaodb["organ"] == "Immune system"]["official gene symbol"]):
    ensembl_gene_ids = list(
        ensembl_gene_id_gene_name_mapping[ensembl_gene_id_gene_name_mapping["Gene name"] == gene][
            "Gene stable ID"
        ]
    )
    marker_ensembl_gene_ids += ensembl_gene_ids


In [ ]:
tad_group_marker_genes = [
    np.any([tad_group_gene in marker_ensembl_gene_ids for tad_group_gene in tad_group.split("_")])
    for tad_group in graph_csv_list[0][0].index
]


In [ ]:
ratio_marker_gene = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    print(phenotype)
    mean_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    mean_not_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[~np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    print(mean_marker)
    print(mean_not_marker)
    ratio_marker_gene.append(mean_marker / mean_not_marker)
np.mean(ratio_marker_gene)


In [ ]:
os.getcwd()


In [ ]:
gene_group_marker_genes = [
    gene in marker_ensembl_gene_ids for gene in graph_csv_genes_list[0][0].index
]


In [ ]:
ratio_marker_gene = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    print(phenotype)
    mean_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    mean_not_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[~np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    print(mean_marker)
    print(mean_not_marker)
    ratio_marker_gene.append(mean_marker / mean_not_marker)
np.mean(ratio_marker_gene)


In [ ]:
ratio_marker_gene = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    print(phenotype)
    mean_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
        - graph_csv_baseline_list[0][phenotype_index].iloc[np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    mean_not_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[~np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
        - graph_csv_baseline_list[0][phenotype_index].iloc[~np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    print(mean_marker)
    print(mean_not_marker)
    ratio_marker_gene.append(mean_marker / mean_not_marker)
np.mean(ratio_marker_gene)


In [ ]:
ratio_marker_gene = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    print(phenotype)
    mean_marker = np.mean(
        graph_csv_genes_list[0][phenotype_index].iloc[np.array(gene_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    mean_not_marker = np.mean(
        graph_csv_genes_list[0][phenotype_index].iloc[~np.array(gene_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    print(mean_marker)
    print(mean_not_marker)
    ratio_marker_gene.append(mean_marker / mean_not_marker)
np.mean(ratio_marker_gene)


In [ ]:
ratio_marker_gene = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    print(phenotype)
    mean_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    mean_not_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[~np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    print(mean_marker)
    print(mean_not_marker)
    ratio_marker_gene.append(mean_marker / mean_not_marker)
np.mean(ratio_marker_gene)


In [ ]:
ratio_marker_gene = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    print(phenotype)
    mean_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    mean_not_marker = np.mean(
        graph_csv_list[0][phenotype_index].iloc[~np.array(tad_group_marker_genes)][
            "predictors_label_valid_roc"
        ]
    )
    print(mean_marker)
    print(mean_not_marker)
    ratio_marker_gene.append(mean_marker / mean_not_marker)
np.mean(ratio_marker_gene)


### Sensitivity analysis - Is one gene of a TAD irrelevant?

In [ ]:
genes_of_interest = [
    "ENSG00000164307",
    "ENSG00000247121",
    "ENSG00000248734",
]
model_paths = [
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/",
]


In [ ]:
for gene_of_interest in genes_of_interest:
    print(gene_of_interest)
    for model_path in model_paths:
        path = f"{model_path}cell_type_vs_rest_res_unp_act_mask_False_flip_False_perturbation_{gene_of_interest}_evaluated_graph_csv.pkl"
        with open(path, "rb") as f:
            perturbation = pickle.load(f)
        print(
            perturbation["expression"]["expression"]["CD14+ Mono"][
                list(graph_csv_list[0][phenotype_index].index).index("_".join(genes_of_interest))
            ]
        )


In [ ]:
for gene_of_interest in genes_of_interest:
    print(gene_of_interest)
    for model_path in model_paths:
        path = f"{model_path}cell_type_vs_rest_res_unp_act_mask_False_flip_False_perturbation_{gene_of_interest}_evaluated_graph_csv.pkl"
        with open(path, "rb") as f:
            perturbation = pickle.load(f)
        print(
            perturbation["expression"]["expression"]["CD16+ Mono"][
                list(graph_csv_list[0][phenotype_index].index).index("_".join(genes_of_interest))
            ]
        )


In [ ]:
phenotype_index = [x for _, x in sorted(zip(phenotypes_labels, phenotypes))].index("CD14+_Mono")


In [ ]:
phenotype_index = [x for _, x in sorted(zip(phenotypes_labels, phenotypes))].index("CD14+_Mono")


In [ ]:
genes_of_interest = ["ENSG00000146192", "ENSG00000164530", "ENSG00000198663"]
model_paths = [
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/",
    "../.experiment_data/primary/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/",
]


In [ ]:
for gene_of_interest in genes_of_interest:
    print(gene_of_interest)
    for model_path in model_paths:
        path = f"{model_path}cell_type_vs_rest_res_unp_act_mask_False_flip_False_perturbation_{gene_of_interest}_evaluated_graph_csv.pkl"
        with open(path, "rb") as f:
            perturbation = pickle.load(f)
        print(
            perturbation["expression"]["expression"]["CD14+ Mono"][
                list(graph_csv_list[0][phenotype_index].index).index("_".join(genes_of_interest))
            ]
        )


In [ ]:
for gene_of_interest in genes_of_interest:
    print(gene_of_interest)
    for model_path in model_paths:
        path = f"{model_path}cell_type_vs_rest_res_unp_act_mask_False_flip_False_perturbation_{gene_of_interest}_evaluated_graph_csv.pkl"
        with open(path, "rb") as f:
            perturbation = pickle.load(f)
        print(
            perturbation["expression"]["expression"]["Normoblast"][
                list(graph_csv_list[0][phenotype_index].index).index("_".join(genes_of_interest))
            ]
        )


In [ ]:
for gene_of_interest in genes_of_interest:
    print(gene_of_interest)
    for model_path in model_paths:
        path = f"{model_path}cell_type_vs_rest_res_unp_act_mask_False_flip_False_perturbation_{gene_of_interest}_evaluated_graph_csv.pkl"
        with open(path, "rb") as f:
            perturbation = pickle.load(f)
        print(
            perturbation["expression"]["expression"]["Erythroblast"][
                list(graph_csv_list[0][phenotype_index].index).index("_".join(genes_of_interest))
            ]
        )


In [ ]:
phenotype_index = [x for _, x in sorted(zip(phenotypes_labels, phenotypes))].index("CD14+_Mono")


In [ ]:
phenotype_index = [x for _, x in sorted(zip(phenotypes_labels, phenotypes))].index("Normoblast")


In [ ]:
phenotype_index = [x for _, x in sorted(zip(phenotypes_labels, phenotypes))].index("Erythroblast")


In [ ]:
go_terms_of_interest = [
    "GO:0002366",
    "GO:0002521",
    "GO:0002429",
    "GO:0002507",
    "GO:0030183",
    "GO:0002377",
    "GO:0045190",
    "GO:0002312",
    "GO:0070670",
    "GO:0019882",
    "GO:0002208",
    "GO:0033151",
    "GO:0050851",
    "GO:0045190",
    "GO:0002312",
    "GO:0002320",
    "GO:0042113",
    "GO:0001923",
    "GO:0001924",
    "GO:0002317",
    "GO:0002431",
    "GO:0030183",
]


In [ ]:
for go_term in go_terms_of_interest:
    if go_term in graph_csv_go_list[0][0].index:
        print(go_term)
        print(obodag[go_term].name)


In [ ]:
phenotype_index = phenotypes.index("Naive_CD20+_B_IGKC-")


In [ ]:
phenotypes.index("Naive_CD20+_B_IGKC+")


In [ ]:
from matplotlib.colors import Normalize, to_hex
from matplotlib.cm import ScalarMappable

norm = Normalize(vmin=-0.5, vmax=0.5)
mappable = ScalarMappable(norm=norm, cmap=cmap)


In [ ]:
go_term = "GO:0002376"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0045321"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0002274"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0050902"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0046649"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0002269"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0006955"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0006959"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0042092"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0002250"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0090716"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0002460"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
go_term = "GO:0002387"
print(graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"])
print(
    to_hex(
        mappable.to_rgba(
            graph_csv_go_list[0][phenotype_index].loc[go_term]["predictors_label_valid_roc"]
        )
    )
)


In [ ]:
scipy.stats.spearmanr(
    graph_csv_go_list[0][26]["predictors_label_valid_roc"],
    graph_csv_go_list[0][27]["predictors_label_valid_roc"],
)


In [ ]:
obodag.keys()


In [ ]:
for go_name in [
    obodag[go_term].name for go_term in graph_csv_go_list[0][0].index if go_term in obodag.keys()
]:
    print(go_name)


## TAD importance vs TAD size (genes per TAD)

In [ ]:
tad_sizes = np.array([len(tad.split('_')) for tad in graph_csv_list[0][0].index])
mean_gain_per_tad = np.mean(np.array(diffs), axis=0)
rho_size, p_size = scipy.stats.spearmanr(tad_sizes, mean_gain_per_tad)
print(f'TAD size vs importance gain: Spearman rho={rho_size:.3f}, p={p_size:.4e}')
set_figure_params('single', 2.2)
fig, ax = plt.subplots()
for s in np.unique(tad_sizes):
    mask = tad_sizes == s
    ax.scatter(np.full(mask.sum(), s) + np.random.normal(0, 0.1, mask.sum()), mean_gain_per_tad[mask], alpha=0.4, s=3, color='#70acc0')
ax.set_xlabel('Number of genes in TAD')
ax.set_ylabel('Mean TAD importance gain')
ax.set_title(f'Spearman ρ = {rho_size:.2f}, p = {p_size:.3e}')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, 'tad_size_vs_importance_gain.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, 'tad_size_vs_importance_gain.svg'), bbox_inches='tight', format='svg')
plt.show()


In [ ]:
rho_size_boots = []
for _ in range(10000):
    idx = np.random.choice(len(mean_gain_per_tad), len(mean_gain_per_tad), replace=True)
    rho_size_boots.append(scipy.stats.spearmanr(tad_sizes[idx], mean_gain_per_tad[idx]).statistic)
print(
    f"TAD size rho CI: [{np.percentile(rho_size_boots, 2.5):.3f}, {np.percentile(rho_size_boots, 97.5):.3f}]"
)


## Marker gene enrichment with bootstrap CIs

In [ ]:
gene_ratios_marker = []
gene_ratios_nonmarker = []
for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    marker_imp = (
        graph_csv_genes_list[0][phenotype_index]
        .iloc[np.array(gene_group_marker_genes)]["predictors_label_valid_roc"]
        .values
    )
    non_marker_imp = (
        graph_csv_genes_list[0][phenotype_index]
        .iloc[~np.array(gene_group_marker_genes)]["predictors_label_valid_roc"]
        .values
    )
    gene_ratios_marker.append(np.mean(marker_imp))
    gene_ratios_nonmarker.append(np.mean(non_marker_imp))

ratios = np.array(gene_ratios_marker) / np.array(gene_ratios_nonmarker)
res = bootstrap((ratios,), np.mean, n_resamples=10000, confidence_level=0.95)
stat, p = wilcoxon(gene_ratios_marker, gene_ratios_nonmarker)
print(
    f"Marker vs non-marker gene importance ratio: mean={np.mean(ratios):.4f}, 95% CI=[{res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f}], Wilcoxon p={p:.4e}"
)


In [ ]:
from scipy.stats import wilcoxon, bootstrap

tad_marker_overall = []
tad_nonmarker_overall = []

for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    imp = graph_csv_list[0][phenotype_index]["predictors_label_valid_roc"]
    marker_mask = np.array(tad_group_marker_genes)
    tad_marker_overall.append(np.mean(imp.iloc[marker_mask]))
    tad_nonmarker_overall.append(np.mean(imp.iloc[~marker_mask]))

tad_marker_overall = np.array(tad_marker_overall)
tad_nonmarker_overall = np.array(tad_nonmarker_overall)
ratios_overall = tad_marker_overall / tad_nonmarker_overall

res_overall = bootstrap((ratios_overall,), np.mean, n_resamples=10000, confidence_level=0.95)
stat_overall, p_overall = wilcoxon(tad_marker_overall, tad_nonmarker_overall)
print(
    f"TAD overall: marker/non-marker ratio = {np.mean(ratios_overall):.4f} "
    f"({(np.mean(ratios_overall)-1)*100:.1f}%), "
    f"95% CI [{(res_overall.confidence_interval.low-1)*100:.1f}%, "
    f"{(res_overall.confidence_interval.high-1)*100:.1f}%], "
    f"Wilcoxon p = {p_overall:.4e}"
)

tad_marker_gain = []
tad_nonmarker_gain = []

for phenotype_index, phenotype in enumerate(
    [x for _, x in sorted(zip(phenotypes_labels, phenotypes))]
):
    gain = (
        graph_csv_list[0][phenotype_index]["predictors_label_valid_roc"]
        - graph_csv_baseline_list[0][phenotype_index]["predictors_label_valid_roc"]
    )
    marker_mask = np.array(tad_group_marker_genes)
    tad_marker_gain.append(np.mean(gain.iloc[marker_mask]))
    tad_nonmarker_gain.append(np.mean(gain.iloc[~marker_mask]))

tad_marker_gain = np.array(tad_marker_gain)
tad_nonmarker_gain = np.array(tad_nonmarker_gain)

abs_diff_gain = tad_marker_gain - tad_nonmarker_gain
res_gain = bootstrap((abs_diff_gain,), np.mean, n_resamples=10000, confidence_level=0.95)
stat_gain, p_gain = wilcoxon(tad_marker_gain, tad_nonmarker_gain)
print(
    f"TAD gain: marker - non-marker absolute diff = {np.mean(abs_diff_gain):.4f}, "
    f"95% CI [{res_gain.confidence_interval.low:.4f}, {res_gain.confidence_interval.high:.4f}], "
    f"Wilcoxon p = {p_gain:.4e}"
)

ratio_gain = tad_marker_gain / tad_nonmarker_gain
res_ratio_gain = bootstrap((ratio_gain,), np.mean, n_resamples=10000, confidence_level=0.95)
print(
    f"TAD gain ratio: {np.mean(ratio_gain):.4f} "
    f"({(np.mean(ratio_gain)-1)*100:.1f}%), "
    f"95% CI [{(res_ratio_gain.confidence_interval.low-1)*100:.1f}%, "
    f"{(res_ratio_gain.confidence_interval.high-1)*100:.1f}%]"
)


# FURTHER CI

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def get_ci_auroc(_graph_csv, go_term):
    activations = _graph_csv.loc[go_term]["activations_test"]
    case_activations = _graph_csv.loc[go_term]["case_activations"]

    n = len(activations)

    case_set = set(map(tuple, case_activations))
    labels = np.array([tuple(row) in case_set for row in activations], dtype=bool)

    X = activations
    y = labels.astype(int)

    clf = LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=100)
    clf.fit(X, y)

    y_score = clf.predict_proba(X)[:, 1]

    auroc = roc_auc_score(y_true=y, y_score=y_score)

    def bootstrap_auc_ci(y_true, y_score, n_boot=200, alpha=0.05, random_state=42):
        rng = np.random.default_rng(random_state)
        y_true = np.asarray(y_true)
        y_score = np.asarray(y_score)
        n = len(y_true)
        boot_aucs = []
        for _ in range(n_boot):
            idx = rng.choice(n, size=n, replace=True)
            if len(np.unique(y_true[idx])) < 2:
                continue
            boot_aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
        lower = np.percentile(boot_aucs, 100 * alpha / 2)
        upper = np.percentile(boot_aucs, 100 * (1 - alpha / 2))
        median = np.median(boot_aucs)
        return median, lower, upper

    median, lower, upper = bootstrap_auc_ci(y, y_score)

    return auroc, median, lower, upper


In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

GENE_GROUPS_MODEL = (
    "../.experiment_data/scratch/experiments_new/"
    "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_"
    "patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_"
    "batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_"
    "gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type"
)
GENE_GROUPS_BASELINE = (
    "../.experiment_data/scratch/experiments/"
    "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_"
    "patscib_False_scarches_False_ec_True_dic_True_ddic_True_"
    "batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_"
    "ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type"
)
GENES_MODEL = (
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_"
    "scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_"
    "scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_"
    "modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/"
    "default/calc_genes_cell_type"
)
GENES_BASELINE = (
    "../.experiment_data/primary/experiments/"
    "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_"
    "patscib_False_scarches_False_ec_True_dic_True_ddic_True_"
    "batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_"
    "ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_genes_cell_type"
)
GENE_GROUPS_DROPOUT_MODEL = (
    "../.experiment_data/primary/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_"
    "scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_"
    "scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_"
    "modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/"
    "default/calc_gene_groups_cell_type/"
)
GENE_GROUPS_DROPOUT_BASELINE = (
    "../.experiment_data/primary/experiments/"
    "sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_"
    "patscib_False_scarches_False_ec_True_dic_True_ddic_True_"
    "batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_"
    "ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/"
)

MODALITY = "expression"

TARGETS = {
    "TAD1": {
        "tad_id": "ENSG00000146192_ENSG00000164530_ENSG00000198663",
        "genes": ["ENSG00000146192", "ENSG00000164530", "ENSG00000198663"],
        "cell_types": [
            ("CD14+_Mono", "CD14+ Mono"),
            ("Normoblast", "Normoblast"),
            ("Erythroblast", "Erythroblast"),
        ],
    },
    "TAD2": {
        "tad_id": "ENSG00000164307_ENSG00000247121_ENSG00000248734",
        "genes": ["ENSG00000164307", "ENSG00000247121", "ENSG00000248734"],
        "cell_types": [
            ("CD14+_Mono", "CD14+ Mono"),
            ("CD16+_Mono", "CD16+ Mono"),
        ],
        "dropout_genes": [
            "ENSG00000164307",
            "ENSG00000247121",
            "ENSG00000248734",
        ],
    },
}

def _load(path):
    graph_csv = pd.read_pickle(path)
    graph_csv.index = ["_".join(sorted(v.split("_"))) for v in graph_csv.index]
    graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
    return graph_csv

def load_tad_model(phenotype):
    return _load(
        f"{GENE_GROUPS_MODEL}/{MODALITY}_cell_type_vs_rest_{phenotype}_"
        f"res_unp_act_mask_False_flip_False_gene_groups_ENCFF173VDJ_"
        f"K562_GRCh38_contact_domains_juicertools_gene_groups_"
        f"evaluated_graph_csv.pkl"
    )

def load_tad_baseline(phenotype):
    return _load(
        f"{GENE_GROUPS_BASELINE}/{MODALITY}_cell_type_vs_rest_{phenotype}_"
        f"res_unp_act_mask_False_flip_False_gene_groups_ENCFF173VDJ_"
        f"K562_GRCh38_contact_domains_juicertools_gene_groups_"
        f"evaluated_graph_csv.pkl"
    )

def load_gene_model(phenotype_label):
    return _load(
        f"{GENES_MODEL}/{MODALITY}_cell_type_vs_rest_{phenotype_label}_"
        f"res_unp_act_mask_False_flip_False_genes_evaluated_graph_csv.pkl"
    )

def load_gene_baseline(phenotype_label):
    return _load(
        f"{GENES_BASELINE}/{MODALITY}_cell_type_vs_rest_{phenotype_label}_"
        f"res_unp_act_mask_False_flip_False_genes_evaluated_graph_csv.pkl"
    )

def load_tad_dropout_auroc(phenotype_label, dropout_gene, tad_id, df_tad_ref, baseline=False):
    '''Perturbation pickles have no phenotype in filename.'''
    base = GENE_GROUPS_DROPOUT_BASELINE if baseline else GENE_GROUPS_DROPOUT_MODEL
    path = (
        f"{base}cell_type_vs_rest_res_unp_act_mask_False_flip_False_"
        f"perturbation_{dropout_gene}_evaluated_graph_csv.pkl"
    )
    with open(path, "rb") as f:
        data = pickle.load(f)
    arr = data[MODALITY][MODALITY][phenotype_label]
    tad_index = list(df_tad_ref.index).index(tad_id)
    return float(arr[tad_index])

def get_ci_auroc(df, feature_id, n_boot=500):
    row = df.loc[feature_id]

    act_col = None
    for candidate in ["activations_test", "activations_valid", "activations"]:
        if candidate in row.index:
            act_col = candidate
            break
    if act_col is None:
        raise KeyError(f"No activations column. Available: {row.index.tolist()}")

    activations = row[act_col]
    case_act = row["case_activations"]

    case_set = set(map(tuple, case_act))
    labels = np.array([tuple(r) in case_set for r in activations], dtype=int)

    if len(np.unique(labels)) < 2:
        raise ValueError(f"Only one class present for {feature_id}")

    clf = LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=100)
    clf.fit(activations, labels)
    y_score = clf.predict_proba(activations)[:, 1]
    auroc = roc_auc_score(labels, y_score)

    rng = np.random.default_rng(42)
    boot = []
    for _ in range(n_boot):
        idx = rng.choice(len(labels), len(labels), replace=True)
        if len(np.unique(labels[idx])) < 2:
            continue
        boot.append(roc_auc_score(labels[idx], y_score[idx]))

    lo, hi = np.percentile(boot, [2.5, 97.5])
    return auroc, np.median(boot), lo, hi

def safe_get_ci(df, feature_id, n_boot=500):
    try:
        return get_ci_auroc(df, feature_id, n_boot)
    except Exception as e:
        print(f"      [skipped: {feature_id}: {e}]")
        return None, None, None, None

results = {}

for tad_key, cfg in TARGETS.items():
    results[tad_key] = {}
    print(f"\n{'='*60}\n  {tad_key}: {cfg['tad_id']}\n{'='*60}")

    for phenotype, phenotype_label in cfg["cell_types"]:
        results[tad_key][phenotype_label] = {}
        print(f"\n  ▸ {phenotype_label}")

        df_tad_m = load_tad_model(phenotype)
        df_tad_b = load_tad_baseline(phenotype)
        df_gene_m = load_gene_model(phenotype_label)
        df_gene_b = load_gene_baseline(phenotype_label)

        tad_id = cfg["tad_id"]
        auroc_m, _, lo_m, hi_m = safe_get_ci(df_tad_m, tad_id)
        auroc_b, _, lo_b, hi_b = safe_get_ci(df_tad_b, tad_id)

        results[tad_key][phenotype_label]["tad_model"] = (auroc_m, lo_m, hi_m)
        results[tad_key][phenotype_label]["tad_baseline"] = (auroc_b, lo_b, hi_b)

        def fmt(v):
            return f"{v:.3f}" if v is not None else "n/a"

        print(f"    TAD model:    {fmt(auroc_m)} (95% CI {fmt(lo_m)}–{fmt(hi_m)})")
        print(f"    TAD baseline: {fmt(auroc_b)} (95% CI {fmt(lo_b)}–{fmt(hi_b)})")
        if auroc_m is not None and auroc_b is not None:
            print(f"    TAD gain:     {auroc_m - auroc_b:.3f}")

        for gene in cfg["genes"]:
            auroc_gm, _, lo_gm, hi_gm = safe_get_ci(df_gene_m, gene)
            auroc_gb, _, lo_gb, hi_gb = safe_get_ci(df_gene_b, gene)

            results[tad_key][phenotype_label][f"{gene}_model"] = (auroc_gm, lo_gm, hi_gm)
            results[tad_key][phenotype_label][f"{gene}_baseline"] = (auroc_gb, lo_gb, hi_gb)

            gain_str = (
                f"{auroc_gm - auroc_gb:.3f}"
                if auroc_gm is not None and auroc_gb is not None
                else "n/a"
            )
            fold_str = (
                f"{auroc_gm/auroc_gb:.2f}x"
                if auroc_gm is not None and auroc_gb is not None and auroc_gb > 0
                else "n/a"
            )

            print(
                f"    {gene}:  "
                f"model {fmt(auroc_gm)} ({fmt(lo_gm)}–{fmt(hi_gm)})  "
                f"baseline {fmt(auroc_gb)} ({fmt(lo_gb)}–{fmt(hi_gb)})  "
                f"gain {gain_str}  fold {fold_str}"
            )

        if "dropout_genes" in cfg:
            print(f"\n    Dropout sensitivity ({phenotype_label}):")
            ci_half = ((hi_m - lo_m) / 2) if hi_m is not None and lo_m is not None else None

            for dropout_gene in cfg["dropout_genes"]:
                try:
                    auroc_p = load_tad_dropout_auroc(
                        phenotype_label, dropout_gene, tad_id, df_tad_m, baseline=False
                    )
                    auroc_p_b = load_tad_dropout_auroc(
                        phenotype_label, dropout_gene, tad_id, df_tad_m, baseline=True
                    )
                    gain = auroc_p - auroc_p_b
                    ci_str = f"±{ci_half:.3f} proxy" if ci_half is not None else "n/a"
                    print(
                        f"      dropout {dropout_gene}: "
                        f"model {auroc_p:.3f} ({ci_str})  "
                        f"baseline {auroc_p_b:.3f}  "
                        f"gain {gain:.3f}"
                    )
                except Exception as e:
                    print(f"      dropout {dropout_gene}: [skipped: {e}]")

print("\nDone.")


## Bias analysis

In [ ]:
mean_gain_per_tad = np.mean(np.array(diffs), axis=0)
hi_c_scores_aligned = hi_c_scores[index_gene_groups]
rho_hic, p_hic = scipy.stats.spearmanr(hi_c_scores_aligned, mean_gain_per_tad)
rho_hic_boots = []
n = len(mean_gain_per_tad)
rng = np.random.default_rng(42)
for _ in range(10000):
    idx = rng.choice(n, n, replace=True)
    rho_hic_boots.append(scipy.stats.spearmanr(hi_c_scores_aligned[idx], mean_gain_per_tad[idx]).statistic)
ci_lo_hic = np.percentile(rho_hic_boots, 2.5)
ci_hi_hic = np.percentile(rho_hic_boots, 97.5)
print(f'Hi-C score vs TAD importance gain:')
print(f'  Spearman rho = {rho_hic:.3f}, p = {p_hic:.4e}')
print(f'  Bootstrap 95% CI: [{ci_lo_hic:.3f}, {ci_hi_hic:.3f}]')
set_figure_params('single', 2.2)
fig, ax = plt.subplots()
ax.scatter(hi_c_scores_aligned, mean_gain_per_tad, alpha=0.4, s=3, color='#70acc0')
ax.set_xlabel('Hi-C contact domain score')
ax.set_ylabel('Mean TAD importance gain')
ax.set_title(f'Spearman ρ = {rho_hic:.3f}, p = {p_hic:.3e}')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, 'hic_score_vs_tad_importance_gain.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, 'hic_score_vs_tad_importance_gain.svg'), bbox_inches='tight', format='svg')
plt.show()


In [ ]:
rng = np.random.default_rng(42)
rna_var = mdata.mod['rna'].var.copy()
print('Available var columns:', rna_var.columns.tolist())
if 'highly_variable_rank' in rna_var.columns:
    hvg_score_col = 'highly_variable_rank'
    hvg_scores = rna_var['highly_variable_rank'].fillna(rna_var['highly_variable_rank'].max() + 1)
    hvg_dict = dict(zip(rna_var['gene_stable_id'], hvg_scores))
    invert = True
elif 'dispersions_norm' in rna_var.columns:
    hvg_score_col = 'dispersions_norm'
    hvg_scores = rna_var['dispersions_norm'].fillna(0)
    hvg_dict = dict(zip(rna_var['gene_stable_id'], hvg_scores))
    invert = False
elif 'variances_norm' in rna_var.columns:
    hvg_score_col = 'variances_norm'
    hvg_scores = rna_var['variances_norm'].fillna(0)
    hvg_dict = dict(zip(rna_var['gene_stable_id'], hvg_scores))
    invert = False
else:
    hvg_score_col = 'mean_expression_proxy'
    mean_expr = np.array(mdata.mod['rna'].X.mean(axis=0)).flatten()
    hvg_dict = dict(zip(rna_var['gene_stable_id'], mean_expr))
    invert = False
print(f'Using column: {hvg_score_col}')
tad_mean_hvg = []
tad_ids_ordered = list(graph_csv_list[0][0].index)
for tad_id in tad_ids_ordered:
    genes_in_tad = tad_id.split('_')
    scores = [hvg_dict.get(g, np.nan) for g in genes_in_tad]
    scores = [s for s in scores if not np.isnan(s)]
    tad_mean_hvg.append(np.mean(scores) if scores else np.nan)
tad_mean_hvg = np.array(tad_mean_hvg)
valid_mask = ~np.isnan(tad_mean_hvg)
print(f'TADs with valid HVG scores: {valid_mask.sum()} / {len(valid_mask)}')
rho_hvg, p_hvg = scipy.stats.spearmanr(tad_mean_hvg[valid_mask], mean_gain_per_tad[valid_mask])
rho_hvg_boots = []
n_valid = valid_mask.sum()
hvg_valid = tad_mean_hvg[valid_mask]
gain_valid = mean_gain_per_tad[valid_mask]
for _ in range(10000):
    idx = rng.choice(n_valid, n_valid, replace=True)
    rho_hvg_boots.append(scipy.stats.spearmanr(hvg_valid[idx], gain_valid[idx]).statistic)
ci_lo_hvg = np.percentile(rho_hvg_boots, 2.5)
ci_hi_hvg = np.percentile(rho_hvg_boots, 97.5)
print(f'\nHVG score ({hvg_score_col}) vs TAD importance gain:')
print(f'  Spearman rho = {rho_hvg:.3f}, p = {p_hvg:.4e}')
print(f'  Bootstrap 95% CI: [{ci_lo_hvg:.3f}, {ci_hi_hvg:.3f}]')
set_figure_params('single', 2.2)
fig, ax = plt.subplots()
ax.scatter(tad_mean_hvg[valid_mask], mean_gain_per_tad[valid_mask], alpha=0.4, s=3, color='#70acc0')
ax.set_xlabel(f'Mean TAD gene variability ({hvg_score_col})')
ax.set_ylabel('Mean TAD importance gain')
ax.set_title(f'Spearman ρ = {rho_hvg:.3f}, p = {p_hvg:.3e}')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, 'hvg_score_vs_tad_importance_gain.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, 'hvg_score_vs_tad_importance_gain.svg'), bbox_inches='tight', format='svg')
plt.show()
print('\n' + '=' * 60)
print('BIAS ANALYSIS SUMMARY')
print('=' * 60)
print(f'TAD size (n genes):    rho={0.01:.3f}, 95% CI [-0.064, 0.085], p=0.756')
print(f'Hi-C domain score:     rho={rho_hic:.3f}, 95% CI [{ci_lo_hic:.3f}, {ci_hi_hic:.3f}], p={p_hic:.3e}')
print(f'HVG score:             rho={rho_hvg:.3f}, 95% CI [{ci_lo_hvg:.3f}, {ci_hi_hvg:.3f}], p={p_hvg:.3e}')


In [ ]:
os.getcwd()


## Negative TAD controls

In [ ]:
mean_gain_per_tad = np.mean(np.array(diffs), axis=0)
bottom_5pct_indices = np.where(mean_gain_per_tad < np.quantile(mean_gain_per_tad, 0.10))[0]
bottom_tads = np.array(list(graph_csv_list[0][0].index))[bottom_5pct_indices]

for tad in bottom_tads[:10]:
    genes = tad.split("_")
    gene_names = [
        (
            ensembl_gene_id_gene_name_mapping[
                ensembl_gene_id_gene_name_mapping["Gene stable ID"] == g
            ]["Gene name"].iloc[0]
            if len(
                ensembl_gene_id_gene_name_mapping[
                    ensembl_gene_id_gene_name_mapping["Gene stable ID"] == g
                ]
            )
            > 0
            else g
        )
        for g in genes
    ]
    print(tad, "→", gene_names)
    print(f"  Mean gain: {mean_gain_per_tad[list(graph_csv_list[0][0].index).index(tad)]:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
tad_ids_ordered = list(graph_csv_list[0][0].index)
top5_indices = np.argsort(mean_gain_per_tad)[::-1][:10]
top5_tad_ids = [tad_ids_ordered[i] for i in top5_indices]

def tad_to_label(tad_id, ensembl_gene_id_gene_name_mapping):
    genes = tad_id.split('_')
    names = []
    for g in genes:
        match = ensembl_gene_id_gene_name_mapping[ensembl_gene_id_gene_name_mapping['Gene stable ID'] == g]['Gene name']
        if len(match) > 0 and isinstance(match.iloc[0], str) and match.iloc[0]:
            names.append(match.iloc[0])
        else:
            names.append(g[:12])
    return ', '.join(names)
top5_labels = [tad_to_label(t, ensembl_gene_id_gene_name_mapping) for t in top5_tad_ids]
cell_type_order = [x for _, x in sorted(zip(phenotypes_labels, phenotypes_labels))]
heatmap_data = pd.DataFrame(np.array(diffs)[:, top5_indices], index=cell_type_order, columns=top5_labels)
set_figure_params('double', 5)
fig, ax = plt.subplots()
sns.heatmap(heatmap_data, cmap=cmap_heat, center=0, vmin=-0.05, vmax=0.2, ax=ax, linewidths=0.3, linecolor='white', cbar_kws={'label': 'TAD importance gain', 'shrink': 0.6})
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=5)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=4.5)
rep_tad_ids_sorted = ['_'.join(sorted(t.split('_'))) for t in ['ENSG00000146192_ENSG00000164530_ENSG00000198663', 'ENSG00000164307_ENSG00000247121_ENSG00000248734']]
for col_idx, tad_id in enumerate(top5_tad_ids):
    if '_'.join(sorted(tad_id.split('_'))) in rep_tad_ids_sorted:
        ax.add_patch(plt.Rectangle((col_idx, 0), 1, len(cell_type_order), fill=False, edgecolor='#c12075', lw=1.5, clip_on=False))
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, 'top10_tad_importance_heatmap.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, 'top10_tad_importance_heatmap.svg'), bbox_inches='tight', format='svg')
plt.show()

def has_named_gene(tad_id, ensembl_gene_id_gene_name_mapping):
    genes = tad_id.split('_')
    for g in genes:
        match = ensembl_gene_id_gene_name_mapping[ensembl_gene_id_gene_name_mapping['Gene stable ID'] == g]['Gene name']
        if len(match) > 0 and match.iloc[0]:
            return True
    return False
named_mask = np.array([has_named_gene(tad_ids_ordered[i], ensembl_gene_id_gene_name_mapping) for i in range(len(tad_ids_ordered))])
bottom5_indices = np.argsort(mean_gain_per_tad * named_mask.astype(float) + (~named_mask).astype(float) * 999)[:10]
bottom5_tad_ids = [tad_ids_ordered[i] for i in bottom5_indices]
bottom5_labels = [tad_to_label(t, ensembl_gene_id_gene_name_mapping) for t in bottom5_tad_ids]
heatmap_data_bottom = pd.DataFrame(np.array(diffs)[:, bottom5_indices], index=cell_type_order, columns=bottom5_labels)
set_figure_params('double', 5)
fig, ax = plt.subplots()
sns.heatmap(heatmap_data_bottom, cmap=cmap_heat, center=0, vmin=-0.05, vmax=0.05, ax=ax, linewidths=0.3, linecolor='white', cbar_kws={'label': 'TAD importance gain', 'shrink': 0.6})
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=5)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=4.5)
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, 'bottom10_tad_importance_heatmap.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, 'bottom10_tad_importance_heatmap.svg'), bbox_inches='tight', format='svg')
plt.show()


In [ ]:
print("TOP 10 TADs by mean importance gain:")
print("=" * 60)
top10_indices = np.argsort(mean_gain_per_tad)[::-1][:10]
for rank, idx in enumerate(top10_indices):
    tad_id = tad_ids_ordered[idx]
    label = tad_to_label(tad_id, ensembl_gene_id_gene_name_mapping)
    print(f"{rank+1:2d}. {label}")
    print(f"    Ensembl: {tad_id}")
    print(f"    Mean gain: {mean_gain_per_tad[idx]:.4f}")
    print()

print("\nBOTTOM 10 TADs by mean importance gain (named genes only):")
print("=" * 60)
named_indices = [
    i
    for i in range(len(tad_ids_ordered))
    if has_named_gene(tad_ids_ordered[i], ensembl_gene_id_gene_name_mapping)
]
named_gains = [(i, mean_gain_per_tad[i]) for i in named_indices]
named_gains_sorted = sorted(named_gains, key=lambda x: x[1])

for rank, (idx, gain) in enumerate(named_gains_sorted[:10]):
    tad_id = tad_ids_ordered[idx]
    label = tad_to_label(tad_id, ensembl_gene_id_gene_name_mapping)
    print(f"{rank+1:2d}. {label}")
    print(f"    Ensembl: {tad_id}")
    print(f"    Mean gain: {gain:.4f}")
    print()
